# Non instructional pretrain llm finetuning on domain specific dataset**

In [ ]:
! pip install -U peft  bitsandbytes transformers accelerate trl PyMuPDF torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


### Non Instructional Fine Tuning Pipeline
* Data Collection
* Cleaning/Filtering
* Spliting/Chunking
* Model Selection and Tokenizer loading
* Tokenization
* Training(Next Token Prediction)
* Infrencing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

## LoRA (Append some external weights to the already trained pretrain weight)

In [ ]:
# libraries
from  transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

### 1. Data Collection and Splitting (Chunking)

In [ ]:
from datasets import Dataset, load_dataset
import re
import fitz

In [ ]:
def extract_text_from_pdf(pdf_path):
  text_blocks = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text=page.get_text("text").strip()
      if text:
        text_blocks.append(text)
  return text_blocks

##### Our own custom data (non instruction data) for domain specific finetuning

In [ ]:
pdf_texts = extract_text_from_pdf("/content/AIDO.pdf")

In [ ]:
pdf_texts[0]

'Toward AI-Driven Digital Organism:\nA System of Multiscale Foundation Models for Predicting,\nSimulating and Programming Biology at All Levels\nLe Song⋆,⋄, Eran Segal⋆,⋄,†, Eric Xing⋆,⋄,‡\n⋆GenBio AI\n⋄Mohamed bin Zayed University of Artificial Intelligence\n†Weizmann Institute of Science\n‡Carnegie Mellon University\n{le.song, eran.segal, eric.xing}@genbio.ai\nAbstract\nWe present a vision of using AI to model and simulate biology and life. Why is it important?\nBecause at the core of medicine, pharmacy, public health, longevity, agriculture and food security,\nenvironmental protection, and clean energy, it is biology at work. Biology in the physical world is\ntoo complex to manipulate and always expensive and risky to tamper with. In this perspective,\nwe layout an engineering viable approach to address this challenge by constructing an AI-Driven\nDigital Organism (AIDO), a system of integrated multiscale foundation models, in a modular,\nconnectable, and holistic fashion to reflect

###2. Data Chunking

In [ ]:
import re
def split_paragraph(pages):
  paragraphs = []
  for page_text in pages:
    # Split on double line breaks
    chunks = re.split(r'\n\s*\n', page_text)
    for chunk in chunks:
      clean = chunk.strip()
      if len(clean)>30:
        paragraphs.append(clean)
  return paragraphs


In [ ]:
paragraphs = split_paragraph(pdf_texts)

In [ ]:
paragraphs

['Toward AI-Driven Digital Organism:\nA System of Multiscale Foundation Models for Predicting,\nSimulating and Programming Biology at All Levels\nLe Song⋆,⋄, Eran Segal⋆,⋄,†, Eric Xing⋆,⋄,‡\n⋆GenBio AI\n⋄Mohamed bin Zayed University of Artificial Intelligence\n†Weizmann Institute of Science\n‡Carnegie Mellon University\n{le.song, eran.segal, eric.xing}@genbio.ai\nAbstract\nWe present a vision of using AI to model and simulate biology and life. Why is it important?\nBecause at the core of medicine, pharmacy, public health, longevity, agriculture and food security,\nenvironmental protection, and clean energy, it is biology at work. Biology in the physical world is\ntoo complex to manipulate and always expensive and risky to tamper with. In this perspective,\nwe layout an engineering viable approach to address this challenge by constructing an AI-Driven\nDigital Organism (AIDO), a system of integrated multiscale foundation models, in a modular,\nconnectable, and holistic fashion to reflec

In [ ]:
data = [{"text": p} for p in paragraphs]

In [ ]:
data

[{'text': 'Toward AI-Driven Digital Organism:\nA System of Multiscale Foundation Models for Predicting,\nSimulating and Programming Biology at All Levels\nLe Song⋆,⋄, Eran Segal⋆,⋄,†, Eric Xing⋆,⋄,‡\n⋆GenBio AI\n⋄Mohamed bin Zayed University of Artificial Intelligence\n†Weizmann Institute of Science\n‡Carnegie Mellon University\n{le.song, eran.segal, eric.xing}@genbio.ai\nAbstract\nWe present a vision of using AI to model and simulate biology and life. Why is it important?\nBecause at the core of medicine, pharmacy, public health, longevity, agriculture and food security,\nenvironmental protection, and clean energy, it is biology at work. Biology in the physical world is\ntoo complex to manipulate and always expensive and risky to tamper with. In this perspective,\nwe layout an engineering viable approach to address this challenge by constructing an AI-Driven\nDigital Organism (AIDO), a system of integrated multiscale foundation models, in a modular,\nconnectable, and holistic fashion 

In [ ]:
from datasets import Dataset, load_dataset

In [ ]:
dataset = Dataset.from_list(data)

In [ ]:
dataset

Dataset({
    features: ['text'],
    num_rows: 32
})

In [ ]:
dataset[0]

{'text': 'Toward AI-Driven Digital Organism:\nA System of Multiscale Foundation Models for Predicting,\nSimulating and Programming Biology at All Levels\nLe Song⋆,⋄, Eran Segal⋆,⋄,†, Eric Xing⋆,⋄,‡\n⋆GenBio AI\n⋄Mohamed bin Zayed University of Artificial Intelligence\n†Weizmann Institute of Science\n‡Carnegie Mellon University\n{le.song, eran.segal, eric.xing}@genbio.ai\nAbstract\nWe present a vision of using AI to model and simulate biology and life. Why is it important?\nBecause at the core of medicine, pharmacy, public health, longevity, agriculture and food security,\nenvironmental protection, and clean energy, it is biology at work. Biology in the physical world is\ntoo complex to manipulate and always expensive and risky to tamper with. In this perspective,\nwe layout an engineering viable approach to address this challenge by constructing an AI-Driven\nDigital Organism (AIDO), a system of integrated multiscale foundation models, in a modular,\nconnectable, and holistic fashion t

###3. Model Selection

#### 3.1 Download modal

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
save_path = "/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T/"


In [ ]:
model.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T/tokenizer_config.json',
 '/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T/tokenizer.json')

#### 3.2 Load Modal from previously saved

In [ ]:
saved_path = "/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
# Loading Model
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(saved_path)
tokenizer = AutoTokenizer.from_pretrained(saved_path)

Loading weights:   0%|          | 0/201 [00:15<?, ?it/s]

###4.  Tokenization

In [ ]:
saved_path = "/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(saved_path)

* Pad token (Used to make all sequence in a btach the same length)
* EOS_token (It means End of Sequence - marks the end of a sentence or chunk )

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token


* truncation=True [If the text is longer than 512 tokens, cut it off (keep only first 512)]
* padding ="max_length" [If the text is shorter tan 512 tokens, pad it with  <pad> tokens upto length]
* max_length= 512 [Find sequence length per training  example(standard size)]


In [ ]:
def tokenize_fn(examples):
  tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
  tokens['labels'] =tokens["input_ids"].copy()
  return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [ ]:
tokenized = tokenized.filter(lambda x: x["input_ids"] is not None)

Filter:   0%|          | 0/32 [00:00<?, ? examples/s]



* tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

{'input_ids':[1,4382,272,98....], 'attention_mask': [1,1,1,....,0,0,0]}

*  Labels
tokens['labels'] =tokens["input_ids"].copy()
This is the key step for causal (unsupervised) language modelling.
It means "The model should try to predict the next token of this same sequence".

*  Outputs
The final dictionary returned looks like this
{
  "input_ids": [...],
  "attention_mask": [...],
  "labels":[...]
}

After that we will get input_ids , attenion mask, labels


In [ ]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 32
})

In [ ]:
tokenized[0]['labels']

[1,
 22578,
 538,
 319,
 29902,
 29899,
 29928,
 374,
 854,
 15918,
 9205,
 1608,
 29901,
 13,
 29909,
 2184,
 310,
 9683,
 10669,
 744,
 10606,
 3382,
 1379,
 363,
 21099,
 919,
 292,
 29892,
 13,
 8942,
 18099,
 322,
 7835,
 4056,
 3457,
 3002,
 472,
 2178,
 21597,
 29879,
 13,
 3226,
 9362,
 229,
 142,
 137,
 29892,
 229,
 142,
 135,
 29892,
 382,
 661,
 6667,
 284,
 229,
 142,
 137,
 29892,
 229,
 142,
 135,
 29892,
 30087,
 29892,
 14713,
 1060,
 292,
 229,
 142,
 137,
 29892,
 229,
 142,
 135,
 29892,
 30892,
 13,
 229,
 142,
 137,
 15462,
 29933,
 601,
 319,
 29902,
 13,
 229,
 142,
 135,
 29924,
 1148,
 2795,
 9016,
 796,
 388,
 287,
 3014,
 310,
 3012,
 928,
 616,
 3159,
 28286,
 13,
 30087,
 4806,
 466,
 4403,
 8907,
 310,
 9327,
 13,
 30892,
 29907,
 2753,
 387,
 347,
 341,
 514,
 265,
 3014,
 13,
 29912,
 280,
 29889,
 21453,
 29892,
 14960,
 29889,
 10199,
 284,
 29892,
 604,
 293,
 29889,
 29916,
 292,
 29913,
 29992,
 1885,
 24840,
 29889,
 1794,
 13,
 9118,
 13,
 4806,


### 5. Training

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/LLM/LLM_NFT_QLORA",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none"
)

In [ ]:

help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    saved_path,
    quantization_config=quant_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(saved_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Loara config
lora_config=LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

In [ ]:
# Loading Quantized LoRA model
q_lora_model =get_peft_model(model, lora_config)

In [ ]:
trainer = Trainer(
    model=q_lora_model,
    args=training_args,
    train_dataset=tokenized
)

In [ ]:
print(tokenized[0])

{'input_ids': [1, 22578, 538, 319, 29902, 29899, 29928, 374, 854, 15918, 9205, 1608, 29901, 13, 29909, 2184, 310, 9683, 10669, 744, 10606, 3382, 1379, 363, 21099, 919, 292, 29892, 13, 8942, 18099, 322, 7835, 4056, 3457, 3002, 472, 2178, 21597, 29879, 13, 3226, 9362, 229, 142, 137, 29892, 229, 142, 135, 29892, 382, 661, 6667, 284, 229, 142, 137, 29892, 229, 142, 135, 29892, 30087, 29892, 14713, 1060, 292, 229, 142, 137, 29892, 229, 142, 135, 29892, 30892, 13, 229, 142, 137, 15462, 29933, 601, 319, 29902, 13, 229, 142, 135, 29924, 1148, 2795, 9016, 796, 388, 287, 3014, 310, 3012, 928, 616, 3159, 28286, 13, 30087, 4806, 466, 4403, 8907, 310, 9327, 13, 30892, 29907, 2753, 387, 347, 341, 514, 265, 3014, 13, 29912, 280, 29889, 21453, 29892, 14960, 29889, 10199, 284, 29892, 604, 293, 29889, 29916, 292, 29913, 29992, 1885, 24840, 29889, 1794, 13, 9118, 13, 4806, 2198, 263, 18551, 310, 773, 319, 29902, 304, 1904, 322, 29611, 4768, 3002, 322, 2834, 29889, 3750, 338, 372, 4100, 29973, 13, 29933, 

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=8, training_loss=2.5651724338531494, metrics={'train_runtime': 46.2178, 'train_samples_per_second': 1.385, 'train_steps_per_second': 0.173, 'total_flos': 203614870044672.0, 'train_loss': 2.5651724338531494, 'epoch': 2.0})

### 6. Infrencing

In [ ]:
model_path ="/content/drive/MyDrive/LLM/LLM_NFT_QLORA/checkpoint-8"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
trained_model=AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
saved_path = "/content/drive/MyDrive/LLM/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(saved_path)

In [ ]:
prompt="AI-Driven Digital Organism (AIDO)"

In [ ]:
input= tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = trained_model.generate(
    **input,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print("\n model Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


 model Output: 

AI-Driven Digital Organism (AIDO)
NVIDIA DGX Station 3: The First Large-Scale NVIDIA-Provided Solution for Mixed-Reality Computing
Raspberry Pi 400: A Portable, Affordable Computer That's Changing the Way We Learn and Create
The Most Compelling New Features in Windows 10 S
How to Enable Vulnerability Scanning for Your AWS Compute Instances Using Terraform
An
